# 1. 参数管理
在选择了架构并设置了超参数后，就进入了训练阶段。此时，我们的目标是找到损失函数最小化的模型参数值。经过训练后，我们将需要使用这些参数来做出未来的预测。此外，有时我们希望提取参数，以便在其他环境中复用它们，并将模型保存下来，以便可以在其他软件中执行。

In [3]:
import torch
from torch import nn

# 具有单隐藏层的多层感知机

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size = (2, 4)) # 两个样本，每个样本四个特征
print(net(X))

tensor([[0.0072],
        [0.0257]], grad_fn=<AddmmBackward0>)


## 1.1参数访问
我们从已有模型中访问参数。当通过Sequential类定义模型时，我们可以通索引访问模型的任意层。

In [4]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.0768, -0.1470, -0.3505,  0.0660, -0.2190, -0.2394,  0.2553, -0.0444]])), ('bias', tensor([0.2816]))])


这个全连接层包含两个参数，分别是该层的权重和偏置。注意，参数名称唯一标识每个参数，即使在包含数百个层的网络中也是如此。


### 1.目标参数
每个参数都表示为参数类的一个实例。要对参数执行任何操作，首先我们需要访问底层的参数值。

In [5]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2816], requires_grad=True)
tensor([0.2816])


参数是复合的对象，包含值、梯度和额外信息。就是我们需要显示参数值的原因。除了参数值，我们还可以访问每个参数的梯度。上面代码中我们没有反向传播所以参数的梯度处于初始状态。

In [6]:
net[2].weight.grad == None

True

### 2.一次性访问所有参数
当我们需要对所有参数执行操作时，逐个访问它们可能会很麻烦。当我们处理复杂的块时，情况可能会变得更复杂，因为我们需要递归整个树来提取每个子块的参数。

In [7]:
# 访问第一个全连接层的参数和访问所有层的参数
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [8]:
# 另一种访问网络参数的方式
net.state_dict()['2.bias'].data

tensor([0.2816])

### 3.从嵌套块收集参数

In [11]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i} ', block1())
    return net
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.0673],
        [-0.0673]], grad_fn=<AddmmBackward0>)

In [12]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0 ): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1 ): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2 ): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3 ): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


因为层是分层嵌套的，所以我们也可以像通过嵌套列表索引那样访问层。下面我们访问第一个主要块中第二个子块的第一层偏置。

In [13]:
rgnet[0][1][0].bias.data

tensor([ 0.4552,  0.2537,  0.1566,  0.2739, -0.3117,  0.4995, -0.0022, -0.0114])

## 1.2参数初始化
深度学习框架提供默认的随机初始化，也允许我们创建自定义的初始化方法，以满足我们通过其他规则初始化权重。 </br>
默认情况下，**PyTorch会根据一个范围均与地初始化权重和偏置矩阵，这个范围是根据输入维度和输出维度计算出的。PyTorch的nn.init模块提供了多种内置初始化方法。**

### 1.内置初始化

In [14]:
# 下面的代码将所有权重参数初始化标准差为0.01的高斯随机变量，且将偏置参数设置为0。
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std = 0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0139,  0.0064, -0.0060, -0.0138]), tensor(0.))

In [17]:
# 还可以将所有参数初始化为给定的常量，如初始化为1
# 下面的代码将所有权重参数初始化标准差为0.01的高斯随机变量，且将偏置参数设置为0。
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [19]:
# 我们还可以对不同块使用不同的初始化方法
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight) # 使用Xavier初始化方法

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42) # 常量42初始化

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.6791,  0.0193, -0.2618, -0.1052])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### 2.自定义初始化
下面，我们使用以下分布为任意泉州i参数w定义初始化方法：
$$
w~\left\{\begin{matrix}
U(5, 10), \ \ \ \ \ \ \ \ 可能性为\frac{1}{4} \\
0, \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ 可能性为\frac{1}{2} \\
U(-10, -5) \ \ \ \ \ 可能性为\frac{1}{4}
\end{matrix}\right.
$$

In [20]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
            """
                将权重初始化为 [-10, 10] 范围内的均匀分布；
                下划线 _ 表示原地操作（直接修改，不返回新张量）
            """
        m.weight.data *= m.weight.data.abs() >= 5 # 保留绝对值≥5的权重，将绝对值<5的权重设为0

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 0.0000, -9.6924,  0.0000,  0.0000],
        [ 8.0266, -0.0000,  8.9199,  5.8539]], grad_fn=<SliceBackward0>)

**值得注意的是我们始终可以直接设置参数**

In [22]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000, -8.6924,  1.0000,  1.0000])

## 1.3参数绑定
有时我们希望在多个层之间共享参数：我们可以定义一个稠密层，然后用这个稠密层的参数设置另一个层的参数。

In [27]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                   shared, nn.ReLU(),
                   shared, nn.ReLU(),
                   nn.Linear(8, 1))

net(X)

print(net[2].weight.data[0] == net[4].weight.data[0])

net[2].weight.data[0, 0] = 100 #确保它们是同一个对象而不是同一个值 

print(net[2].weight.data[0] == net[4].weight.data[0])

print(net[2].weight.data[0] == shared.weight.data[0])
print(shared.weight.data[0] == net[4].weight.data[0])

shared.weight.data[0, 0] += 5

print(net[2].weight.data[0] == shared.weight.data[0])
print(shared.weight.data[0] == net[4].weight.data[0])


tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


当层和层之间的参数绑定时，它们的值不仅相等，而且由相同的张量表示。因此，如果我们改变其中一个参数，另一个参数也会改变。

**当参数绑定时，梯度也是模型的参数，所以反向传播时第三个神经网络net[2]和第五个神经网络net[4]的梯度会加在一起。**